<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_3/bonus/pandas_data_analysis/note_bonus_extra_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Бонус М3 — додаткові датасети: реальні дані, реальні проблеми

Три ноутбуки уроку вчать pandas, matplotlib і seaborn/Plotly на цінах WFP. Тут — ще чотири **справжні** джерела, кожне зі своєю «хворобою»:

| Розділ | Датасет | Чого вчить |
|---|---|---|
| 1 | зарплати за регіонами (data.gov.ua, Держстат, 2000–2018) | «брудний» CSV: cp1251, `;`, тисячі через пробіл, `…` і `х` замість чисел → `melt` |
| 2 | зарплата проти ціни хліба | об'єднання двох джерел (`merge`) і похідний показник |
| 3 | показники Світового банку (1960–2025) | «довгий» формат → `pivot` |
| 4 | щоденна активність портів (2019–2026) | часові ряди: `resample`, «до/після» |
| 5 | ціни WFP у 71 країні (2026) | великий файл: `usecols`, фільтр, рейтинг |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Теорія — у книзі курсу: [Бонус. Pandas](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m3/bonus_pandas/) і [довідник з аналізу даних](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m3/pandas/data_analytics/).

In [ ]:
import importlib.util, os, subprocess, sys

LESSON = "module_3/bonus/pandas_data_analysis"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"
if not os.path.isdir("data"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))
missing = [m for m in ("pandas", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "data"
print("файлів у data/:", len(os.listdir(DATA_DIR)), "| pandas", pd.__version__)

---
## 1. Зарплати: «брудний» CSV з data.gov.ua

Файл Держстату відкриваємо як є — і дивимось, що всередині:

In [ ]:
SALARY = os.path.join(DATA_DIR, "101-serednomisiachna-nominalna-zarobitna-plata-za-regionami-u-rozrakhunku-na-odnogo-shtatnogo-pr.csv")
with open(SALARY, "rb") as f:
    print(f.read(120))

Байти `\xca\xe2…` — це **cp1251** (Windows-кодування кирилиці), не UTF-8. А ще: кожен рядок обгорнутий у `;…;;;;`, усередині — коми, тисячі відділені **пробілом** (`1 041`), а замість чисел трапляються `…` (немає даних) і `х` (показник не рахувався).

План очищення: прочитати як один стовпець між `;` → розбити по комі → перший рядок — заголовок → `melt` у «довгий» формат → числа.

### Вправа 1

Прочитай файл (`encoding="cp1251"`, `sep=";"`, `header=None`), візьми **другий** стовпець (індекс 1) без порожніх рядків і розбий кожен рядок по комі (`.str.split(",")`). Результат (Series списків) — у `rows`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
raw = pd.read_csv(SALARY, encoding="cp1251", sep=";", header=None)
rows = raw[1].dropna().str.split(",")
# END SOLUTION

assert rows.iloc[0][1:4] == ["2000", "2005", "2006"], "перший рядок — заголовок з роками"
assert rows.iloc[1][:3] == ["Україна", "230", "806"]
print(rows.iloc[0])
print(rows.iloc[2])
print("✅ Вправа 1 пройдена")

### Вправа 2

Збери «широку» таблицю `wide`: рядки з тією самою кількістю елементів, що й заголовок, і непорожнім першим числом; перший стовпець назви `"регіон"` (прибери пробіли, зокрема нерозривний `\xa0`, — `.str.strip()`), решта — роки.

In [ ]:
header = rows.iloc[0]
# YOUR CODE HERE
# BEGIN SOLUTION
data = [row for row in rows.iloc[1:] if len(row) == len(header) and row[1] != ""]
wide = pd.DataFrame(data, columns=["регіон"] + header[1:])
wide["регіон"] = wide["регіон"].str.strip()
# END SOLUTION

assert wide.shape == (29, 16), wide.shape
assert wide["регіон"].iloc[-2] == "м. Київ"
wide.tail(3)

### Вправа 3

`melt` → «довгий» формат (`регіон`, `рік`, `зарплата`). Числа: прибери пробіли, `…` і `х` стануть `NaN` через `pd.to_numeric(..., errors="coerce")`; `рік` — `int`. Результат — у `salary`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
salary = wide.melt(id_vars="регіон", var_name="рік", value_name="зарплата")
salary["зарплата"] = pd.to_numeric(salary["зарплата"].str.replace(" ", ""), errors="coerce")
salary["рік"] = salary["рік"].astype(int)
# END SOLUTION

assert len(salary) == 29 * 15 and salary["зарплата"].isna().sum() == 21, "21 пропуск — це «…» і «х»"
top2018 = salary.query("рік == 2018").nlargest(3, "зарплата")
print(top2018.to_string(index=False))
print("✅ Вправа 3 пройдена")

**🔮 Прогноз:** У скільки разів зарплата в Києві 2018 року більша за найнижчу серед областей?

<details>
<summary>Відповідь</summary>

Порахуй: `salary.query('рік == 2018')` → `max / min`. Київ — 13 542 грн, найнижча — Тернопільська, 6 969 грн: у 1,94 раза.

</details>

---
## 2. Зарплата проти ціни хліба: скільки буханок?

Дві таблиці з різних джерел: зарплати (Держстат, рік) і ціни (WFP, місяць). Зводимо до спільного ключа — **року** — і рахуємо похідний показник.

Для ціни беремо офіційний WFP «National Average» житнього хліба (2014–2017 — лише ці роки є в даних на національному рівні); для зарплати — рядок `Україна*` (без тимчасово окупованих територій — так рахує Держстат з 2010 року).

### Вправа 4

Середня ціна буханки `Bread (rye)` по роках (`admin1` порожній, `currency == "UAH"`) — у `bread`; об'єднай з зарплатою `Україна*` за роком (`merge`, `how="inner"`) і додай стовпець `буханок = зарплата / ціна`. Результат — `power`.

In [ ]:
food = pd.read_csv(os.path.join(DATA_DIR, "wfp_food_prices_ukr.csv"), parse_dates=["date"])
# YOUR CODE HERE
# BEGIN SOLUTION
bread = (
    food.query("commodity == 'Bread (rye)' and admin1.isna() and currency == 'UAH'")
    .assign(рік=lambda d: d["date"].dt.year)
    .groupby("рік", as_index=False)["price"].mean()
    .rename(columns={"price": "ціна_хліба"})
)
power = bread.merge(salary.query("регіон == 'Україна*'")[["рік", "зарплата"]], on="рік", how="inner")
power["буханок"] = (power["зарплата"] / power["ціна_хліба"]).round(0)
# END SOLUTION

assert list(power["рік"]) == [2014, 2015, 2016, 2017]
print(power.round(2).to_string(index=False))
print("✅ Вправа 4 пройдена")

**🔮 Прогноз:** Зарплата за 2014–2017 зросла вдвічі. Чи зросла вдвічі купівельна спроможність у «буханках»?

<details>
<summary>Відповідь</summary>

Ні: зарплата виросла з 3 480 до 7 104 грн (×2,04), а буханок — 619 → 458 → 511 → 595: у 2015 хліб подорожчав на 63%, а зарплата лише на 21%. Номінальне зростання ≠ реальне; щоб порівнювати роки, гроші треба «знецінювати» цінами.

</details>

---
## 3. Показники Світового банку: «довгий» формат → `pivot`

`climate-change_ukr.csv` — 48 показників Світового банку для України, кожен рядок = (показник, рік, значення). Для аналізу зручніше «широка» таблиця: рядок — рік, стовпець — показник.

### Вправа 5

`pivot` за `Year` (індекс) і `Indicator Code` (стовпці) → `wb`. Знайди рік з найбільшим населенням (`SP.POP.TOTL`) — у `peak_year`.

In [ ]:
wb_long = pd.read_csv(os.path.join(DATA_DIR, "climate-change_ukr.csv"))
# YOUR CODE HERE
# BEGIN SOLUTION
wb = wb_long.pivot(index="Year", columns="Indicator Code", values="Value").sort_index()
peak_year = int(wb["SP.POP.TOTL"].idxmax())
# END SOLUTION

assert wb.shape[1] == wb_long["Indicator Code"].nunique()
assert peak_year == 1993
print("населення:", wb.loc[[1991, 1993, 2000, 2022, 2024], "SP.POP.TOTL"].map("{:,.0f}".format).to_dict())
print("✅ Вправа 5 пройдена")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
(wb["SP.POP.TOTL"] / 1e6).plot(ax=ax, color="#3b82c4", linewidth=2)
ax.set_title("Населення України, млн (Світовий банк)"); ax.set_xlabel("Рік"); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

---
## 4. Порти: щоденні дані → місяці й роки

`ukraine-daily-port-activity-data-and-shipment-estimates.csv` — щоденні заходи суден і оцінки імпорту/експорту (тонни) по 16 портах, 2019–2026.

### Вправа 6

Зроби `date` індексом і `resample("YE")` — суми `portcalls`, `export`, `import` по роках → `yearly`. Експорт 2022 року порівняно з 2021 — у відсотках (`drop_pct`, ціле).

In [ ]:
ports = pd.read_csv(os.path.join(DATA_DIR, "ukraine-daily-port-activity-data-and-shipment-estimates.csv"), parse_dates=["date"])
# YOUR CODE HERE
# BEGIN SOLUTION
yearly = ports.set_index("date").resample("YE")[["portcalls", "export", "import"]].sum()
yearly.index = yearly.index.year
drop_pct = round(100 * (1 - yearly.loc[2022, "export"] / yearly.loc[2021, "export"]))
# END SOLUTION

assert drop_pct == 63
print((yearly[["export", "import"]] / 1e6).round(1).rename(columns=lambda c: c + ", млн т").to_string())
print(f"експорт 2022 проти 2021: −{drop_pct}%")
print("✅ Вправа 6 пройдена")

In [ ]:
monthly = ports.set_index("date").resample("ME")["export"].sum() / 1e6
fig, ax = plt.subplots(figsize=(11, 3.5))
monthly.plot(ax=ax, color="#2e7d32", linewidth=1.8)
ax.axvline(pd.Timestamp("2022-02-24", tz="UTC"), color="#c62828", linestyle="--", label="24.02.2022")
ax.set_title("Експорт через порти України, млн т на місяць"); ax.set_xlabel(""); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

**🔮 Прогноз:** 2023 рік — найгірший за експортом у даних. Чому 2024-й знову вдвічі більший?

<details>
<summary>Відповідь</summary>

Дані цього не пояснюють — лише показують. Контекст (морський коридор з осені 2023) — зовнішнє знання. Аналітик відділяє «що в даних» від «чому» (див. довідник: діагностична аналітика).

</details>

---
## 5. Ціни у світі: великий файл без зайвого

`wfp_food_prices_global_2026.csv` — 32 МБ, 245 тис. рядків, 71 країна. Читаємо лише потрібні стовпці (`usecols`) — у рази менше пам'яті.

### Вправа 7

Прочитай файл з `usecols=["countryiso3", "commodity", "unit", "pricetype", "usdprice"]`. Для цукру (`commodity == "Sugar"`, `unit == "KG"`, `pricetype == "Retail"`) — медіанна ціна в USD по країнах, від дешевшої до дорожчої → `sugar`. Місце України в рейтингу (1 = найдешевше) — у `ukr_rank`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
world = pd.read_csv(os.path.join(DATA_DIR, "wfp_food_prices_global_2026.csv"),
                    usecols=["countryiso3", "commodity", "unit", "pricetype", "usdprice"])
sugar = (world.query("commodity == 'Sugar' and unit == 'KG' and pricetype == 'Retail'")
         .groupby("countryiso3")["usdprice"].median().sort_values())
ukr_rank = int(sugar.index.get_loc("UKR")) + 1
# END SOLUTION

assert len(sugar) == 40
print(f"країн: {len(sugar)} | Україна: {ukr_rank} місце, ${sugar['UKR']:.2f}/кг | медіана світу: ${sugar.median():.2f}")
print("найдешевше:", sugar.head(3).round(2).to_dict())
print("найдорожче:", sugar.tail(3).round(2).to_dict())
print(f"пам'ять: {world.memory_usage(deep=True).sum() / 1e6:.0f} МБ у DataFrame")
print("✅ Вправа 7 пройдена")

---
## Самоперевірка

1. Як зрозуміти, що файл у cp1251, і що буде, якщо прочитати його як UTF-8?
2. Навіщо `melt` для таблиці зарплат і `pivot` для таблиці Світового банку — це протилежні операції?
3. Чому «буханок на зарплату» — кращий показник, ніж сама зарплата?
4. Що робить `resample("YE")` і чим він відрізняється від `groupby(date.dt.year)`?
5. Навіщо `usecols` для великого файлу?

<details>
<summary>Відповіді</summary>

1. Байти `\xca\xe2…` не складаються в UTF-8; `pd.read_csv` без `encoding` впаде з `UnicodeDecodeError` або покаже «кракозябри».
2. Так: `melt` — «широка» → «довга», `pivot` — навпаки. Обирай форму під задачу: групування й графіки — довга, порівняння стовпців — широка.
3. Він прибирає інфляцію: показує, що можна **купити**, а не скільки гривень.
4. Групує за періодами часу (кінець року) на `DatetimeIndex`; для щоденних даних результат той самий, але `resample` вміє й місяці, тижні, квартали й заповнює порожні періоди.
5. Читаються лише потрібні стовпці — менше пам'яті й часу.

</details>

## Далі

- Книга курсу: [Бонус. Pandas: аналіз даних, графіки і Dash](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m3/bonus_pandas/).
- Дашборд: `cd dash_API && python app.py` → `http://localhost:8055`.